# Allpass/Comb Filtering
Allpass filters are very important in digital signal processing. A basic first-order allpass filter allows all frequencies to pass through without altering their magnitudes, but it introduces a phase shift for all frequencies. It can be altered to create lowpass, highpass, and bandpass filters, and it has many audio signal processing applications.

Below is an implementation of an allpass, lowpass, and highpass filter based upon Jan Wilczek's Python alpass filter code (2).

In [11]:
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import IPython.display as ipd
import librosa
import soundfile as sf
from pathlib import Path

filename = '/content/tomsdiner_mono.wav'
diner, srate = librosa.load(filename)
ipd.Audio(diner, rate=srate)


In [19]:
# set up allpass coefficient and filter

def a1_coefficient(breakfreq, samplerate):
    tan = np.tan(np.pi * breakfreq/samplerate)
    return (tan-1)/(tan+1)

def allpass(inputsig, breakfreq, samplerate):
    # output array initialization
    allpass_out = np.zeros_like(inputsig)

    # inner 1-sample buffer initialization
    dn_1 = 0

    for n in range(inputsig.shape[0]):
        # compute allpass coefficient
        a1 = a1_coefficient(breakfreq[n], samplerate)

        # allpass difference equation
        allpass_out[n] = a1 * inputsig[n] + dn_1

        # store value in buffer for use in next iteration
        dn_1 = inputsig[n] - a1 * allpass_out[n]

    return allpass_out

def highpass(inputsig, cutoff_freq, samplerate, amplitude=1.0):
    # multiply by -1 to invert phase
    allpass_out = -1 * allpass(inputsig, cutoff_freq, samplerate)
    filter_out = inputsig + allpass_out

    # scale magnitude to prevent clipping & apply amplitude
    filter_out *= 0.5
    filter_out *= amplitude

    return filter_out

def lowpass(inputsig, cutoff_freq, samplerate, amplitude=1.0):

    allpass_out = allpass(inputsig, cutoff_freq, samplerate)
    filter_out = inputsig + allpass_out

    # scale magnitude to prevent clipping & apply amplitude
    filter_out *= 0.5
    filter_out *= amplitude

    return filter_out

def toms_allpass():
    og_filename = '/content/tomsdiner_mono.wav'
    tomsdiner, samplerate = sf.read(og_filename)
    inputsig = tomsdiner

    samplerate = 44100
    duration_sec = 5
    amplitude = 1.0
    high_cutoff_freq = np.geomspace(10000, 20, inputsig.shape[0])
    low_cutoff_freq = np.geomspace(5000, 20, inputsig.shape[0])

    high_out = highpass(inputsig, high_cutoff_freq, samplerate, amplitude)
    low_out = lowpass(inputsig, low_cutoff_freq, samplerate, amplitude)

    toms_highpass = "highpass_tomsdiner.wav"
    sf.write(toms_highpass, high_out, samplerate)
    ipd.display(ipd.Audio(high_out, rate=samplerate))

    toms_lowpass = "lowpass_tomsdiner.wav"
    sf.write(toms_lowpass, low_out, samplerate)
    ipd.display(ipd.Audio(low_out, rate=samplerate))

toms_allpass()

# References
1. https://thewolfsound.com/allpass-filter/$0

2. https://thewolfsound.com/allpass-based-lowpass-and-highpass-filters/$0